# Vectorización TF-IDF y partición de datos

**Objetivo:** Convertir el texto lematizado en características TF-IDF y crear splits estratificados para entrenamiento, validación y prueba.

**Fecha:** 2026-07-04

**Autor:** Copilot

## Índice

1. Qué hace el script
2. Funciones y clases clave
3. Gráficos y artefactos
4. Ejemplo reducido
5. Cómo reproducir
6. Resumen de artefactos

## Qué hace el script

El script asigna etiquetas con VADER, construye un vectorizador TF-IDF con max_features=20000, ngram_range=(1, 2) y min_df=5, y produce particiones estratificadas con random_state=42. También calcula top features globales y por clase para entender qué términos dominan la señal.

## Funciones y clases usadas

- **ensure_nltk_resources**
- **load_data**
- **vadar_label**
- **build_vectorizer**
- **top_features_global**
- **top_features_by_label**

## Gráficos y artefactos

- Se guarda artifacts/tfidf_vectorizer.pkl.
- Se guardan artifacts/X_tfidf.npz y artifacts/y.npy.
- Se produce artifacts/train_val_test_split_info.txt.

## Ejemplo reducido: etiquetado VADER y vectorización básica

In [8]:
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.sentiment.vader import SentimentIntensityAnalyzer

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)
INPUT_FILE = PROCESSED_DIR / 'sample_5000_lemmatized.csv'
df = pd.read_csv(INPUT_FILE, encoding='utf-8').head(200)
analyzer = SentimentIntensityAnalyzer()
compounds = df['text_lemma'].fillna('').astype(str).apply(lambda x: analyzer.polarity_scores(x)['compound'])
df['compound'] = compounds
df['label'] = df['compound'].apply(lambda x: 'positive' if x >= 0.05 else 'negative' if x <= -0.05 else 'neutral')
print(df[['text_lemma', 'compound', 'label']].head(10).to_string(index=False))
vectorizer = TfidfVectorizer(max_features=200, ngram_range=(1, 2), min_df=2)
X = vectorizer.fit_transform(df['text_lemma'].fillna(''))
print('shape TF-IDF:', X.shape)


                                                                                                                                                                                                                                                                                           text_lemma  compound    label
jon jones be the goat it be so unbelievably hard what he do he become champ at 23 and stay champion until today he fight the good lightweight champion in their prime he be a double champ with multiple title defense lead in the history in title defense basically undefeated congrat to jon jones    0.9363 positive
                                                                                                                                                                                      you can sense the aggression from jon do not touch I no friendly face off he do not want tom to be there at all   -0.4874 negative
                                                             

## Ejemplo reducido: mostrar top features

In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

texts = ['great fight', 'bad loss', 'great win', 'bad performance']
vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1, 2), min_df=1)
X = vectorizer.fit_transform(texts)
features = pd.DataFrame({'feature': vectorizer.get_feature_names_out(), 'avg_tfidf': X.mean(axis=0).A1}).sort_values('avg_tfidf', ascending=False).head(10)
print(features.to_string(index=False))


        feature  avg_tfidf
            bad   0.243467
          great   0.243467
       bad loss   0.154404
bad performance   0.154404
          fight   0.154404
    great fight   0.154404
      great win   0.154404
           loss   0.154404
    performance   0.154404
            win   0.154404


## Cómo reproducir

python src/step4_vectorize_and_split.py

Este notebook es la entrada directa al entrenamiento del modelo porque define las características que se alimentan a PyTorch.

In [10]:
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
ARTIFACTS_DIR.mkdir(exist_ok=True)
print('Resumen del notebook')
print('- notebook:', Path.cwd().name)
print('- artefactos disponibles:', len(list(ARTIFACTS_DIR.glob('*'))))
print('- timestamp:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))


Resumen del notebook
- notebook: notebooks
- artefactos disponibles: 66
- timestamp: 2026-07-07 21:31:53
